In [1]:
import os
import math
import string
from collections import defaultdict

In [2]:
connector_words = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are',
    'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

In [3]:
def clean_word(word):
    word = word.lower()
    word = word.strip(string.punctuation)

    if word.endswith('structures'):
        word = 'structure'
    elif word.endswith('stacks'):
        word = 'stack'
    elif word.endswith('applications'):
        word = 'application'

    return word

In [4]:
def build_inverted_index(folder_path):
    inverted_index = defaultdict(list)
    document_word_count = {}
    document_words = {}

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)

        with open(filepath, 'r', encoding='utf-8') as file:
            text = file.read()

        words = text.split()
        cleaned_words = []

        for idx, word in enumerate(words):
            cleaned = clean_word(word)

            if cleaned and cleaned not in connector_words:
                cleaned_words.append(cleaned)
                inverted_index[cleaned].append((filename, idx + 1))

        document_word_count[filename] = len(cleaned_words)
        document_words[filename] = cleaned_words

    return inverted_index, document_word_count, document_words

In [5]:
folder_path = 'data/webpages'
inverted_index, document_word_count, document_words = build_inverted_index(folder_path)

In [6]:
word = 'data'

if word in inverted_index:
    print('Pages containing word:', word)
    print(inverted_index[word])
else:
    print('No webpage contains word', word)

Pages containing word: data
[('stackoverflow', 13), ('stack_cprogramming', 3), ('stack_cprogramming', 17), ('stack_cprogramming', 98), ('stack_cprogramming', 113), ('stack_datastructure_wiki', 3), ('stack_datastructure_wiki', 24), ('stack_datastructure_wiki', 102), ('stack_datastructure_wiki', 232), ('stack_datastructure_wiki', 421)]


In [7]:
def compute_tfidf(word, document_name, inverted_index, document_words):
    total_documents = len(document_words)

    word_count_in_doc = document_words[document_name].count(word)
    total_words_in_doc = len(document_words[document_name])

    tf = word_count_in_doc / total_words_in_doc

    docs_containing_word = set([doc for doc, pos in inverted_index[word]])
    idf = math.log(total_documents / len(docs_containing_word))

    return tf * idf

In [8]:
query_word = 'data'

if query_word in inverted_index:
    pages = set([doc for doc, pos in inverted_index[query_word]])

    scores = []
    for page in pages:
        score = compute_tfidf(query_word, page, inverted_index, document_words)
        scores.append((page, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    print(scores)

[('stackoverflow', 0.02492052530550599), ('stack_cprogramming', 0.014671824422289242), ('stack_datastructure_wiki', 0.012571184872213705)]
